# Backtest Statistics

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]

from src.backtesting.backtest_statistics import (
    ClassificationScores,
    Efficiency,
    GeneralCharacteristics,
    ImplementationShortfall,
    Performance,
    Runs,
)

result_dir = PROJECT_ROOT / "data/backtest_results"
returns = pd.read_parquet(
    result_dir / "event_strategy_returns.parquet"
).sort_index()
holdout = returns[returns["partition"].eq("holdout")].copy()
strategies = ["primary_only", "meta_filtered"]
elapsed_years = (
    holdout.index.max() - holdout.index.min()
).total_seconds() / (365.25 * 24 * 60 * 60)
events_per_year = len(holdout) / elapsed_years


## General Characteristics

- **Purpose:** Describe the holdout span, directional exposure, and relationship to the underlying event return.
- **Settings:** Flat positions are excluded from the long ratio; correlation uses each strategy's net event return against `raw_return`.
- **Data:** Use holdout event-level positions and returns; the artifact has no AUM, dollar positions, or continuous target-position series.
- **Decision:** Do not compute capacity, leverage, dollar-position size, bet frequency, holding period, or annualized turnover from unavailable inputs.

In [2]:
start, end = GeneralCharacteristics.time_range(holdout)
general_characteristics = pd.DataFrame.from_dict(
    {
        strategy: {
            "start": start,
            "end": end,
            "ratio_of_longs": GeneralCharacteristics.ratio_of_longs(
                holdout[f"{strategy}_position"]
            ),
            "correlation_to_underlying": (
                GeneralCharacteristics.correlation_to_underlying(
                    holdout[f"{strategy}_net_return"],
                    holdout["raw_return"],
                )
            ),
        }
        for strategy in strategies
    },
    orient="index",
)
general_characteristics.index.name = "strategy"
display(general_characteristics)


,start,end,ratio_of_longs,correlation_to_underlying
strategy,,,,
primary_only,2025-10-20 13:30:01.016243+00:00,2025-12-18 14:30:00.344027+00:00,0.772727,-0.587255
meta_filtered,2025-10-20 13:30:01.016243+00:00,2025-12-18 14:30:00.344027+00:00,0.500000,-0.729461


## Performance

- **Purpose:** Compare compounded and annualized holdout returns together with the frequency and size of winning and losing events.
- **Settings:** Annualization uses the observed event frequency over the elapsed holdout span; hit statistics include only events with a nonzero strategy position.
- **Data:** Use the holdout event-return columns, which contain returns rather than dollar PnL.
- **Decision:** Do not call `pnl` or `pnl_from_long_positions`, because their names would give return sums an incorrect dollar interpretation.

In [3]:
wealth = {}
performance_rows = []
for strategy in strategies:
    gross = holdout[f"{strategy}_gross_return"]
    cost = holdout[f"{strategy}_total_cost"]
    net = holdout[f"{strategy}_net_return"]
    bet_returns = net[holdout[f"{strategy}_position"].ne(0.0)]
    wealth[strategy] = (1.0 + net).cumprod()
    performance_rows.append(
        {
            "strategy": strategy,
            "events": len(net),
            "active_bets": len(bet_returns),
            "gross_return_sum": gross.sum(),
            "total_cost_sum": cost.sum(),
            "net_return_sum": net.sum(),
            "compound_net_return": wealth[strategy].iloc[-1] - 1.0,
            "annualized_rate_of_return": (
                Performance.annualized_rate_of_return(
                    net,
                    periods_per_year=events_per_year,
                )
            ),
            "hit_ratio": Performance.hit_ratio(bet_returns),
            "average_hit": Performance.average_return_from_hits(
                bet_returns
            ),
            "average_miss": Performance.average_return_from_misses(
                bet_returns
            ),
        }
    )

performance = pd.DataFrame(performance_rows).set_index("strategy")
display(performance)


,events,active_bets,gross_return_sum,total_cost_sum,net_return_sum,compound_net_return,annualized_rate_of_return,hit_ratio,average_hit,average_miss
strategy,,,,,,,,,,
primary_only,22,22,0.007725,0.0,0.007725,0.005767,0.036217,0.50,0.009900,-0.009198
meta_filtered,22,4,0.038044,0.0,0.038044,0.038046,0.259858,0.75,0.014027,-0.004036


## Runs

- **Purpose:** Measure whether gains, losses, and event timing are concentrated and summarize the depth and duration of drawdowns.
- **Settings:** HHI uses only events with a nonzero strategy position and groups their timing by calendar month (`freq="ME"`); percentile statistics use `q=0.95`; drawdowns are return ratios from compounded wealth.
- **Data:** Calculate descriptive statistics from the 22 holdout events and their compounded wealth paths.
- **Decision:** Allow HHI or time-under-water to be `NaN` when the sample cannot define the requested statistic.

In [4]:
runs_rows = []
for strategy in strategies:
    net = holdout[f"{strategy}_net_return"]
    bet_returns = net[holdout[f"{strategy}_position"].ne(0.0)]
    drawdown = Runs.drawdown(wealth[strategy])
    time_under_water = Runs.time_under_water(wealth[strategy])
    runs_rows.append(
        {
            "strategy": strategy,
            "hhi_positive_returns": Runs.hhi_positive_returns(
                bet_returns
            ),
            "hhi_negative_returns": Runs.hhi_negative_returns(
                bet_returns
            ),
            "hhi_time_between_bets": Runs.hhi_time_between_bets(
                bet_returns
            ),
            "maximum_drawdown": (
                drawdown.max() if not drawdown.empty else np.nan
            ),
            "percentile_drawdown_95": Runs.percentile_drawdown(
                wealth[strategy], q=0.95
            ),
            "maximum_time_under_water": (
                time_under_water.max()
                if not time_under_water.empty
                else np.nan
            ),
            "percentile_time_under_water_95": (
                Runs.percentile_time_under_water(
                    wealth[strategy], q=0.95
                )
            ),
        }
    )

runs = pd.DataFrame(runs_rows).set_index("strategy")
display(runs)


,hhi_positive_returns,hhi_negative_returns,hhi_time_between_bets,maximum_drawdown,percentile_drawdown_95,maximum_time_under_water,percentile_time_under_water_95
strategy,,,,,,,
primary_only,0.143019,0.048111,0.076446,0.048374,0.047049,0.060583,0.060583
meta_filtered,0.707700,NaN,NaN,0.004036,0.004036,NaN,NaN


## Implementation Shortfall

- **Purpose:** Relate recorded execution costs to the normalized two-way turnover and gross strategy return.
- **Settings:** Each completed event contributes entry plus exit turnover, `2 × abs(position)`; the upstream artifact was generated with `cost=0 bp/side`.
- **Data:** Use recorded positions, gross returns, and total costs; separate broker fees and dollar PnL are unavailable.
- **Decision:** Treat total cost as the available slippage proxy, omit unsupported dollar metrics, and allow a zero cost denominator to produce `NaN`.

In [5]:
shortfall_rows = []
for strategy in strategies:
    gross = holdout[f"{strategy}_gross_return"]
    cost = holdout[f"{strategy}_total_cost"]
    two_way_turnover = 2.0 * holdout[f"{strategy}_position"].abs()
    shortfall_rows.append(
        {
            "strategy": strategy,
            "normalized_two_way_turnover": two_way_turnover.sum(),
            "average_slippage_per_turnover": (
                ImplementationShortfall.average_slippage_per_turnover(
                    cost, two_way_turnover
                )
            ),
            "return_on_execution_costs": (
                ImplementationShortfall.return_on_execution_costs(
                    gross, cost
                )
            ),
        }
    )

implementation_shortfall = pd.DataFrame(shortfall_rows).set_index(
    "strategy"
)
display(implementation_shortfall)


,normalized_two_way_turnover,average_slippage_per_turnover,return_on_execution_costs
strategy,,,
primary_only,44.0,0.0,NaN
meta_filtered,4.2,0.0,NaN


## Efficiency

- **Purpose:** Compare return per unit of variability, active return relative to the underlying event return, and the probability that Sharpe exceeds zero.
- **Settings:** Risk-free return and PSR benchmark Sharpe are zero; annualization uses observed holdout event frequency; information ratio uses `raw_return` as the event-aligned benchmark.
- **Data:** Use event-aligned holdout strategy and underlying returns; CPCV paths are repeated evaluations rather than independent strategy trials.
- **Decision:** Do not force CPCV paths into `deflated_sharpe_ratio`, because the required trial-Sharpe input is unavailable.

In [6]:
efficiency_rows = []
for strategy in strategies:
    net = holdout[f"{strategy}_net_return"]
    efficiency_rows.append(
        {
            "strategy": strategy,
            "sharpe_ratio": Efficiency.sharpe_ratio(net),
            "annualized_sharpe": Efficiency.annualized_sharpe_ratio(
                net, periods_per_year=events_per_year
            ),
            "information_ratio": Efficiency.information_ratio(
                net,
                holdout["raw_return"],
                periods_per_year=events_per_year,
            ),
            "probabilistic_sharpe_ratio": (
                Efficiency.probabilistic_sharpe_ratio(
                    net, benchmark_sharpe_ratio=0.0
                )
            ),
        }
    )

efficiency = pd.DataFrame(efficiency_rows).set_index("strategy")
display(efficiency)


,sharpe_ratio,annualized_sharpe,information_ratio,probabilistic_sharpe_ratio
strategy,,,,
primary_only,0.025456,0.296978,0.772316,0.547310
meta_filtered,0.213616,2.492079,1.715998,0.964641


## Classification Scores

- **Purpose:** Evaluate the frozen primary direction classifier and meta action classifier on the holdout labels.
- **Settings:** All scores use the stored event sample weights; positive labels are `1`; negative log loss uses the complete probability vectors for primary labels `[-1, 1]` and meta labels `[0, 1]`.
- **Data:** Use the saved holdout labels, predictions, probability vectors, and sample weights.
- **Decision:** Report accuracy, precision, recall, F1, and negative log loss because the stored artifact directly supports them.

In [7]:
classification_inputs = {
    "primary": {
        "y_true": holdout["direction_label"],
        "y_pred": holdout["primary_side"],
        "y_pred_proba": pd.DataFrame(
            {
                -1: 1.0 - holdout["primary_probability"],
                1: holdout["primary_probability"],
            },
            index=holdout.index,
        ),
        "labels": [-1, 1],
    },
    "meta": {
        "y_true": holdout["meta_label"],
        "y_pred": holdout["meta_action"],
        "y_pred_proba": pd.DataFrame(
            {
                0: 1.0 - holdout["meta_probability"],
                1: holdout["meta_probability"],
            },
            index=holdout.index,
        ),
        "labels": [0, 1],
    },
}

classification_columns = {}
for model, inputs in classification_inputs.items():
    y_true = inputs["y_true"]
    y_pred = inputs["y_pred"]
    classification_columns[model] = {
        "accuracy": ClassificationScores.accuracy(
            y_true, y_pred, sample_weight=holdout["sample_weight"]
        ),
        "precision": ClassificationScores.precision(
            y_true, y_pred, sample_weight=holdout["sample_weight"]
        ),
        "recall": ClassificationScores.recall(
            y_true, y_pred, sample_weight=holdout["sample_weight"]
        ),
        "f1": ClassificationScores.f1_score(
            y_true, y_pred, sample_weight=holdout["sample_weight"]
        ),
        "negative_log_loss": ClassificationScores.negative_log_loss(
            y_true,
            inputs["y_pred_proba"],
            labels=inputs["labels"],
            sample_weight=holdout["sample_weight"],
        ),
    }

classification = pd.DataFrame(classification_columns)

statistics = (
    performance
    .join(general_characteristics)
    .join(runs)
    .join(implementation_shortfall)
    .join(efficiency)
)
cumulative_returns = pd.DataFrame(wealth).sub(1.0)

display(statistics)
display(classification)
display(cumulative_returns)


,events,active_bets,gross_return_sum,total_cost_sum,net_return_sum,compound_net_return,annualized_rate_of_return,hit_ratio,average_hit,average_miss,...,percentile_drawdown_95,maximum_time_under_water,percentile_time_under_water_95,normalized_two_way_turnover,average_slippage_per_turnover,return_on_execution_costs,sharpe_ratio,annualized_sharpe,information_ratio,probabilistic_sharpe_ratio
strategy,,,,,,,,,,,,,,,,,,,,,
primary_only,22,22,0.007725,0.0,0.007725,0.005767,0.036217,0.50,0.009900,-0.009198,...,0.047049,0.060583,0.060583,44.0,0.0,NaN,0.025456,0.296978,0.772316,0.547310
meta_filtered,22,4,0.038044,0.0,0.038044,0.038046,0.259858,0.75,0.014027,-0.004036,...,0.004036,NaN,NaN,4.2,0.0,NaN,0.213616,2.492079,1.715998,0.964641


,primary,meta
accuracy,0.527854,0.756303
precision,0.582671,0.951404
recall,0.464720,0.567302
f1,0.517054,0.710781
negative_log_loss,-0.718969,-1.029729


,primary_only,meta_filtered
event_start,,
2025-10-20 13:30:01.016243+00:00,-0.005158,0.000000
2025-10-21 13:30:00.492827+00:00,0.000952,0.000000
2025-10-21 18:26:04.665410+00:00,-0.010596,0.000000
2025-10-21 19:56:25.842495+00:00,-0.020114,0.000000
2025-10-22 13:30:01.356738+00:00,-0.028839,0.000000
2025-10-22 17:54:34.360531+00:00,-0.034537,0.000000
2025-10-23 15:40:47.934536+00:00,-0.030814,0.000000
2025-10-27 13:32:09.300647+00:00,-0.024911,0.002436
2025-10-29 13:30:00.132475+00:00,-0.019787,0.002436
